# Aufgabe 0 · Kompatibilität und Integration
Eine Schichtleiterin möchte wissen, welche Wartung bei jeder Messung von Maschine M1 zuletzt begonnen hatte und welche Temperatur in der nächsten Stunde zu erwarten ist. Baut dafür aus vier Dateien eine belastbare Zeitreihe.

Arbeitet in [diesem Notebook](task.ipynb). Die [Aufgabenstellung](README.md) erklärt die Quellen. Ergänzt die Codezellen und haltet eure Entscheidungen direkt darunter fest.

## A · Was bedeutet eine Zeile?
Die Startzelle lädt die vier CSV-Dateien. Prüft Datentypen, Ebenen und Zeitbereiche. Welche Schlüssel identifizieren eine Messung, eine Maschine, einen Wartungsbeginn und einen Referenzwert? Wie häufig misst M1? Welche Einheiten und Zeitzonen gehen aus den Dateien hervor?

In [ ]:
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# Funktioniert im Aufgabenordner und bei einem Start aus der Repository-Wurzel.
for basis in (Path.cwd(), *Path.cwd().parents):
    for ordner in (basis, basis / "tasks" / "00_compatibility_and_integration"):
        daten = ordner / "data"
        if (daten / "zeitreihe_roh.csv").is_file():
            break
    else:
        continue
    break
else:
    raise FileNotFoundError("Aufgabenordner mit den vier CSV-Dateien nicht gefunden.")

messung = pd.read_csv(daten / "zeitreihe_roh.csv", parse_dates=["timestamp"])
stamm = pd.read_csv(daten / "maschinenstamm.csv", parse_dates=["commissioned"])
wartung = pd.read_csv(daten / "wartung.csv", parse_dates=["maintenance_start"])
ziel = pd.read_csv(daten / "prognose_zielreihe.csv", parse_dates=["timestamp"])

schluessel = ["machine_id", "timestamp"]
signale = ["temp_raw", "vibration_raw"]
messung = messung.sort_values(schluessel).reset_index(drop=True)
for name, tabelle in {"Messungen": messung, "Stamm": stamm, "Wartung": wartung, "Referenz": ziel}.items():
    print(f"{name} ({len(tabelle)} Zeilen)")
    display(tabelle.head(2))

In [ ]:
# Prüft mit duplicated(), isna() und diff() Schlüssel, Lücken und Messraster.
# Vergleicht die zeitliche Abdeckung von messung und ziel.

**Euer Befund:** Derselbe Wert in `machine_id` bedeutet … Eine offene fachliche Annahme ist …

## B · Was war um 08:45 Uhr bekannt?
Vergleicht für **beide Signale** Forward Fill, Backward Fill und zeitbasierte lineare Interpolation. Beginnt mit jeder Methode bei denselben Rohwerten und rechnet je Maschine. Behaltet die Rohwerte sowie ein Kennzeichen für ersetzte Werte. Zeigt die Temperatur von 08:30 bis 09:15 Uhr. Welche Schätzung wäre um 08:45 Uhr im laufenden Betrieb möglich gewesen?

In [ ]:
# Ausgangspunkt für euren Vergleich. Erst nach Maschine und Zeit sortieren.
vergleich = messung.copy()
# Ergänzt pro Signal drei Varianten sowie ein Flag für die ursprüngliche Lücke.
# Tipp: Für interpolate(method="time") benötigt ihr einen DatetimeIndex.

**Eure Entscheidung:** Für das historische Schichtprotokoll verwenden wir … Für die Live-Anzeige wählen wir …

## C · Welche Wartung gehört zur Messung?
Verknüpft zuerst die Messungen mit dem Maschinenstamm. Garantiert, dass jede Messung genau einmal erhalten bleibt. Ordnet danach jeder Messung den **letzten Wartungsbeginn derselben Maschine bis und mit Messzeitpunkt** zu. Prüft M1 um 10:45 und um 11:00 Uhr. Was würde ein Join allein auf `machine_id` mit der Zeilenzahl machen?

In [ ]:
# 1. Left Join mit stamm. Tipp: validate="many_to_one".
# 2. Zeit-Join mit wartung. Welche Richtung benötigt pd.merge_asof?
# 3. Kontrolliert die Anzahl Messungen, die Schlüssel und die beiden Grenzfälle.

**Euer Befund:** Was lässt sich aus `maintenance_start` über den Betriebszustand der Maschine ableiten?

## D · Rückblick und Vorhersage
**Rückblick.** Verwendet die interpolierte Temperatur von M1 aus der integrierten Tabelle. Glättet die Reihe mit einem nachlaufenden Mittel über drei Messungen (`min_periods=1`). Verknüpft die Schätzung mit `temp_clean_target` anhand von Maschine und Zeitpunkt. Berechnet den mittleren absoluten Fehler (MAE) und zeichnet Schätzung und Referenz im gemeinsamen Zeitfenster.

In [ ]:
# Bildet die Reihe von M1 aus eurer integrierten Tabelle.
# Bildet rolling(3, min_periods=1).mean(), dann den Join mit der Referenz.
# MAE = Mittelwert der absoluten Differenz. Zeigt auch die Anzahl der Vergleichspaare.

**Vorhersage.** Die Hilfsfunktion berechnet zwei Prognosen für vier Viertelstunden. Die erste schreibt die letzte Temperatur fort, die zweite verlängert den linearen Trend der letzten vier Viertelstunden. Übergebt die ausschliesslich vorwärts gefüllte Temperatur von M1 mit Zeitindex. Testet die Funktion zuerst mit Daten bis **11:45 Uhr** und vergleicht beide Prognosen mit den vier späteren Referenzwerten. Erstellt dann mit Daten bis **17:45 Uhr** die Vorschau für 18:00 bis 18:45 Uhr. Für welche dieser Aussagen könnt ihr einen Prognosefehler berechnen?

In [ ]:
def prognose(historie):
    # Eingabe: lückenloses 15-Minuten-Raster einer Maschine, nur bis zum Prognoseursprung.
    fenster = historie.sort_index().tail(4)
    schritt = pd.Timedelta(minutes=15)
    if len(fenster) != 4 or fenster.isna().any() or not fenster.index.to_series().diff().iloc[1:].eq(schritt).all():
        raise ValueError("Vier aufeinanderfolgende, verfügbare Viertelstunden erforderlich.")
    steigung, achsenabschnitt = np.polyfit(np.arange(4), fenster.to_numpy(), 1)
    zeiten = pd.date_range(fenster.index[-1] + schritt, periods=4, freq=schritt, name="timestamp")
    return pd.DataFrame({
        "Letzter Wert": fenster.iloc[-1],
        "Linearer Trend": achsenabschnitt + steigung * np.arange(4, 8),
    }, index=zeiten)

In [ ]:
# test = prognose(... bis einschliesslich 11:45 Uhr ...)
# Richtet die vier Referenzwerte auf test.index aus und berechnet den MAE je Methode.
# Erstellt dieselben beiden Prognosen nochmals für die Stunde nach 17:45 Uhr.

### Decision Log
1. Welche Prüfung hat euch vor einem falschen Join geschützt?
2. Welche Informationen hättet ihr bei einer Prognose um 11:45 Uhr versehentlich aus der Zukunft verwenden können?
3. Lasst euch von einer KI einen Wartungs-Join vorschlagen. Prüft die Antwort an den zwei Grenzfällen aus Teil C. Welche Regel würdet ihr übernehmen oder ändern?
4. Welche Prognose würdet ihr der Schichtleiterin zeigen und welche Unsicherheit würdet ihr nennen?